## Import DuckDB

In [ ]:
# pip install duckdb
# poetry add duckdb
# uv add duckdb

In [ ]:
import duckdb

## Inital Look at the Data

In [ ]:
DATA_PATH = '../data/land_registry/pp-*.csv'

In [ ]:
duckdb.sql(
    f"""
    SELECT *
    FROM read_csv('{DATA_PATH}', header = false)
    """
)

## Add Column Names

Unfortunately the CSV file does not include a header row, so we specify these.

[How to access HM Land Registry Price Paid Data](https://www.gov.uk/guidance/about-the-price-paid-data) web site provides useful information to help us do this.  Here is the key information extracted from that site:

In [ ]:
land_registry_data = duckdb.sql(
    f"""
    SELECT
        column00 AS id,
        column01 AS price,
        column02 AS date,
        column03 AS postcode,
        column04 AS property_type,
        column05 AS old_new,
        column06 AS duration,
        column07 AS paon,
        column08 AS saon,
        column09 AS street,
        column10 AS locality,
        column11 AS town_city,
        column12 AS district,
        column13 AS county,
        column14 AS ppd_category,
        column15 AS record_type
    FROM read_csv('{DATA_PATH}', header = false)
    """
)

In [ ]:
type(land_registry_data)

## DuckDBPyRelation?

A DuckDBPyRelation is best thought of as a lazy query (or query plan) rather than the data itself.

- Lazy evaluation
- Composable
- Reusable

LINQ expression in C# or a Spark DataFrame - it's a description of what to compute, not the computed result.

In [ ]:
# Displaying forces materialisation:
land_registry_data

## Inspect Data

In [ ]:
duckdb.sql("DESCRIBE FROM land_registry_data")

## Inspect Property Types

In [ ]:
land_registry_data = duckdb.sql(
    """
    SELECT * REPLACE (
        CASE property_type
            WHEN 'D' THEN 'Detached'
            WHEN 'S' THEN 'Semi-Detached'
            WHEN 'T' THEN 'Terraced'
            WHEN 'F' THEN 'Flat'
            WHEN 'O' THEN 'Other'
            ELSE property_type
        END AS property_type
    )
    FROM land_registry_data
    """
)

In [ ]:
duckdb.sql(
    """
    SELECT property_type, COUNT(*) AS count
    FROM land_registry_data
    GROUP BY property_type
    """
)

In [ ]:
land_registry_data = duckdb.sql(
    """
    SELECT * FROM land_registry_data
    WHERE property_type != 'Other'
    """
)

## Add Year Feature

In [ ]:
land_registry_data = duckdb.sql(
    """
    SELECT *, year(date) AS year
    FROM land_registry_data
    """
)

## View Data

In [ ]:
duckdb.sql(
    """
    SELECT id, date, year, property_type, price
    FROM land_registry_data
    LIMIT 10
    """
)

## Generate Insights

In [ ]:
annual_price_by_property_type = duckdb.sql(
    """
    SELECT
        year,
        property_type,
        MEDIAN(price) AS median_price
    FROM land_registry_data
    GROUP BY ALL
    ORDER BY year, property_type
    """
)

In [ ]:
annual_price_by_property_type

## Query Plan

In [ ]:
annual_price_by_property_type.explain()

## Visualise the results

In [ ]:
annual_price_by_property_type.pl()

In [ ]:
import plotly.express as px

In [ ]:
px.line(
    annual_price_by_property_type.pl(),
    x="year",
    y="median_price",
    color="property_type",
    color_discrete_sequence=["#8CC600", "black", "darkgrey", "#EE7423"],
    markers=True,
    title="Median Price by Year and Property Type",
    width=800,
    height=600,
)

## Write to File

In [ ]:
from pathlib import Path
Path("../data/price_paid_insights").mkdir(parents=True, exist_ok=True)

duckdb.sql(
    """
    COPY annual_price_by_property_type
    TO '../data/price_paid_insights/annual_price_by_property_type.parquet'
    (FORMAT PARQUET)
    """
)